Inserts additional records into the base sessions/ advert impressions table between the last record date and current timeperiod

Sessions have been de-duped for dates taking the first occurence as the session 'ground truth'

Sessions are currently only web based sessions

Advert Clicks/Impressions currently filtered for ShoppingBag only and having a likely advertid recorded

In [0]:
## Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:

dbutils.widgets.text(name="catalog_schema_prefix", defaultValue="marketingdata_dev.claire_wilsonbarnes", label="catalog_schema_prefix")

In [0]:
%sql
---Also include prior day as some spessions span over 2 days and result in 2 records -we still want to take the initial session 

CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix|| '.pctr_sessions_interim') AS ( 
WITH cte_max_record AS (
    SELECT 
        max(date) AS latest_date
    FROM 
         IDENTIFIER(:catalog_schema_prefix|| '.pctr_sessions')
)
, cte_sessions AS (
SELECT 
  s.device 
    , CASE WHEN s.device = 'Mobile' THEN 'Mobile'
            WHEN s.device= 'Desktop' THEN 'Desktop'
            ELSE 'Other' END as device_simple
    , s.geocountry
    ,  CASE WHEN s.geocountry IN ('United Kingdom', 'Ireland', 'Jersey', 'Isle of Man', 'Guernsey') THEN 'UK & Ireland'
            WHEN m.segment_name IS NOT NULL THEN m.segment_name
            ELSE 'Other' END AS geocountry_simple
    , s.channel
    , CASE WHEN s.channel IN ('Paid Search', 'Organic Search', 'Paid Social' , 'Organic Social', 'Direct' , 'Email', 'Referral' ,  'SMS') THEN  s.channel 
            WHEN s.channel regexp '^.*(Paid).*$' THEN 'Paid Other'
            WHEN s.channel regexp '^.*(Organic).*$' THEN 'Organic Other'
            ELSE 'Other' END AS channel_simple
    , c.account_number
    , s.UniqueVisitID 
    , s.Timeonsite_seconds
    , c.account_number IS NOT NULL AS customer_filter
    , c.gender
    , RANK() OVER (PARTITION BY s.UniqueVisitID ORDER BY s.DATE ASC) AS session_rank 
    , RANK() OVER( PARTITION BY c.account_number 
                    ORDER BY s.date DESC, s.visitstarthour DESC
                     -- Tiebreakers - if multiple sessions on same day/ hour
                    ,s.UniqueVisitID DESC) AS session_order
    , Dayofweek(s.date) AS session_dow
    , s.date
FROM 
    cte_max_record AS mx,
    marketingdata_prod.warehouse.bq_sessions_next_uk AS s 
    -- Want all sessions regardless of if we can match customer
    LEFT JOIN marketingdata_prod.warehouse.rpid_with_accounts AS rpid 
        ON rpid.roamingprofileid=s.RPID
    LEFT JOIN marketingdata_prod.warehouse.svoccust AS c 
        ON c.account_number =rpid.account_number 
        AND c.countrycode='GB'
        AND c.client='NEXT'
    LEFT JOIN marketingdata_prod.search.nov_country_mapping AS m 
        ON m.country_name=s.geocountry
WHERE 
    --- MUST INCLUDE existing last run date as IT could have a session run over 2 days - which will cause duplicated records so we 
    s.date >= mx.latest_date AND s.date < CURRENT_DATE
)
,cte_app_sessions AS (
SELECT
    COALESCE(s.Device , 'App') AS device
    , CASE WHEN s.device = 'Mobile' THEN 'Mobile'
            WHEN s.device= 'Desktop' THEN 'Desktop'
            ELSE 'Other' END as device_simple
    , s.geocountry
    ,  CASE WHEN s.geocountry IN ('United Kingdom', 'Ireland', 'Jersey', 'Isle of Man', 'Guernsey') THEN 'UK & Ireland'
            WHEN m.segment_name IS NOT NULL THEN m.segment_name
            ELSE 'Other' END AS geocountry_simple
    , s.channel
    , CASE WHEN s.channel IN ('Paid Search', 'Organic Search', 'Paid Social' , 'Organic Social', 'Direct' , 'Email', 'Referral' ,  'SMS') THEN  s.channel 
            WHEN s.channel regexp '^.*(Paid).*$' THEN 'Paid Other'
            WHEN s.channel regexp '^.*(Organic).*$' THEN 'Organic Other'
            ELSE 'Other' END AS channel_simple
    , c.account_number
    , s.UniqueVisitID 
    , s.Timeonsite_seconds
    , c.account_number IS NOT NULL AS customer_filter
    , c.gender
    , RANK() OVER (PARTITION BY s.UniqueVisitID ORDER BY s.DATE ASC) AS session_rank 
    , RANK() OVER( PARTITION BY c.account_number 
                    ORDER BY s.date DESC, s.visitstarthour DESC
                     -- Tiebreakers - if multiple sessions on same day/ hour
                    ,s.UniqueVisitID DESC) AS session_order
    , Dayofweek(s.date) AS session_dow
    , s.date
FROM 
    cte_max_record AS mx,
    marketingdata_prod.warehouse.bq_sessions_next_uk_pp AS s 
    -- Want all sessions regardless of if we can match customer
    LEFT JOIN marketingdata_prod.warehouse.rpid_with_accounts AS rpid 
        ON rpid.roamingprofileid=s.RPID
    LEFT JOIN marketingdata_prod.warehouse.svoccust AS c 
        ON c.account_number =rpid.account_number 
        AND c.countrycode='GB'
        AND c.client='NEXT'
    LEFT JOIN marketingdata_prod.search.nov_country_mapping AS m 
        ON m.country_name=s.geocountry
WHERE 
    --- MUST INCLUDE existing last run date as IT could have a session run over 2 days - which will cause duplicated records so we 
    s.date >= mx.latest_date AND s.date < CURRENT_DATE
  )
  SELECT
    *
  FROM
    cte_sessions
  WHERE
    session_rank = 1
  UNION 
  SELECT
    *
  FROM
    cte_app_sessions
  WHERE
    session_rank = 1
);


In [0]:
%sql

MERGE INTO IDENTIFIER(:catalog_schema_prefix|| '.pctr_sessions') AS t
USING IDENTIFIER(:catalog_schema_prefix || '.pctr_sessions_interim') AS f
ON ( t.uniquevisitID = f.uniquevisitID )
-- 1. If a match is found, update the existing row
WHEN MATCHED THEN 
    UPDATE SET 
      t.device=f.device
    , t.geocountry=f.geocountry
    , t.channel=f.channel
    , t.device_simple=f.device_simple
    , t.geocountry_simple=f.geocountry_simple
    , t.channel_simple=f.channel_simple
    , t.account_number=f.account_number
    , t.UniqueVisitID = f.UniqueVisitID
    , t.Timeonsite_seconds=f.Timeonsite_seconds
    , t.customer_filter=f.customer_filter
    , t.gender= f.gender
    , t.session_rank =f.session_rank
    , t.session_order=f.session_order
    , t.session_dow=f.session_dow
    , t.date=f.date
-- 2. If no match is found, insert a brand new row
WHEN NOT MATCHED THEN
    INSERT (     device
    , geocountry
    , channel
      ,device_simple
    , geocountry_simple
    , channel_simple
    , account_number
    , UniqueVisitID 
    , Timeonsite_seconds
    , customer_filter
    , gender
    , session_rank 
    , session_order
    , session_dow
    , date )
    VALUES (
        f.device
    , f.geocountry
    , f.channel
      , f.device_simple
    , f.geocountry_simple
    , f.channel_simple
    , f.account_number
    , f.UniqueVisitID 
    , f.Timeonsite_seconds
    , f.customer_filter
    , f.gender
    , f.session_rank 
    , f.session_order
    , f.session_dow
    , f.date);


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_ad_clicks_impressions_base_interim') AS (
SELECT 
      s.UniqueVisitID
    , s.account_number
    , a.date
    , a.timestamp
    , DAYOFWEEK(a.date) AS dow
    , weekofyear(a.date) AS woy 
    , s.geocountry 
    , s.channel
    , s.device 
    , s.gender
    , s.geocountry_simple
    , s.channel_simple
    , s.device_simple 
    , a.Level2 As AdvertID
    , split_part(a.Level2 , '_',1) AS pot 
    -- , split_part(a.Level2 , '_',2) AS campaign 
    -- Temporary logic to append the campaign suffixes
    , CASE WHEN lower(a.Level2) regexp '^.*(younger).*$' THEN split_part(a.Level2 , '_',2)||'_Y'
        WHEN lower(a.Level2) regexp '^.*(older).*$' THEN split_part(a.Level2 , '_',2)||'_O'
        WHEN lower(a.Level2) regexp '^.*(toddler).*$' THEN split_part(a.Level2 , '_',2)||'_T'
        WHEN lower(a.Level2) regexp '^.*(baby).*$' THEN split_part(a.Level2 , '_',2)||'_N'
        WHEN lower(a.Level2) regexp '^.*(teen).*$' THEN split_part(a.Level2 , '_',2)||'_TE'
        ELSE split_part(a.Level2 , '_',2) END campaign 
    , REGEXP_EXTRACT(a.Level2, '^.*_(V[1-9])_.*$', 1) AS versionnumber
    , a.PagePath 
    , a.action 
FROM 
-- Only additional records to add 
    IDENTIFIER(:catalog_schema_prefix|| '.pctr_sessions_interim') AS s
    INNER JOIN marketingdata_prod.warehouse.bq_actions_next_uk AS a
        ON s.UniqueVisitID = a.UniqueVisitID
        AND a.action IN ('Banner Impression - Next Ads', 'Banner Click - Next Ads')
        -- Currently filtered for shopping bag and to ensu 
        AND a.PagePath ='/shoppingbag'
        AND a.Level2 regexp "^P"
WHERE
    -- Only insert upto the prior days records
    a.date::date < CURRENT_DATE
);


In [0]:
%sql

MERGE INTO IDENTIFIER(:catalog_schema_prefix|| '.pctr_ad_clicks_impressions_base') AS t
USING IDENTIFIER(:catalog_schema_prefix || '.pctr_ad_clicks_impressions_base_interim') AS f
ON ( t.uniquevisitID = f.uniquevisitID 
    AND t.timestamp = f.timestamp
    ANd t.action=f.action )
-- 1. If a match is found, update the existing row
WHEN MATCHED THEN
    UPDATE SET 
     t.UniqueVisitID=f.UniqueVisitID
    ,t.account_number=f.account_number
    , t.date=f.date
    , t.timestamp=f.timestamp
    , t.dow=f.dow
    , t.woy =f.woy
    , t.geocountry =f.geocountry
    , t.channel=f.channel
    , t.device =f.device
    , t.gender=f.gender
    , t.geocountry_simple=f.geocountry_simple
    , t.channel_simple=f.channel_simple
    , t.device_simple =f.device_simple
    , t.AdvertID=f.AdvertID
    , t.pot =f.pot
    , t.campaign =f.campaign
    , t.versionnumber=f.versionnumber
    , t.PagePath =f.PagePath
    , t.action =f.action
-- 2. If no match is found, insert a brand new row
WHEN NOT MATCHED THEN
    INSERT (     UniqueVisitID
    ,account_number
    , date
    , timestamp
    , dow
    , woy 
    , geocountry 
    , channel
    , device 
    , gender
    , geocountry_simple
    , channel_simple
    , device_simple 
    , AdvertID
    , pot 
    , campaign 
    , versionnumber
    , PagePath 
    , action  )
    VALUES (
        f.UniqueVisitID
    ,f.account_number
    , f.date
    , f.timestamp
    , f.dow
    , f.woy 
    , f.geocountry 
    , f.channel
    , f.device 
    , f.gender
    , f.geocountry_simple
    , f.channel_simple
    , f.device_simple 
    , f.AdvertID
    , f.pot 
    , f.campaign 
    , f.versionnumber
    , f.PagePath 
    , f.action );


In [0]:
%sql
DROP TABLE IDENTIFIER(:catalog_schema_prefix|| '.pctr_sessions_interim') ;
DROP TABLE IDENTIFIER(:catalog_schema_prefix || '.pctr_ad_clicks_impressions_base_interim');